# HomeTask 8: Heart Disease Dataset
https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset?resource=download

## Fields:
* `0  -` age (ordered, keep numerical)
* `1  -` sex (binary - hopefully!)
* `2  -` chest pain type (4 values - categorical)
* `3  -` resting blood pressure (numerical)
* `4  -` serum cholestoral in mg/dl (numerical)
* `5  -` fasting blood sugar > 120 mg/dl (binary)
* `6  -` resting electrocardiographic results (values 0,1,2 - categorical)
* `7  -` maximum heart rate achieved (numerical)
* `8  -` exercise induced angina (binary)
* `9  -` oldpeak = ST depression induced by exercise relative to rest (numerical)
* `10 -` the slope of the peak exercise ST segment (categorical)
* `11 -` number of major vessels (0-3) colored by flourosopy (categorical)
* `12 -` thal: 0 = normal; 1 = fixed defect; 2 = reversable defect (categorical)
* `13 -` target (binary)

In [19]:
# metadata
PROJECT_FOLDER = '..'          # notebooks folder is supposed to be inside project - thus, just go up!

DATA_FOLDER = 'data'           # inside project
DATASET_NAME = 'heart.csv'     # inside data

MODELS_FOLDER = 'models'       # inside project
CONFIG_FOLDER = 'config'       # inside project
PREPROC_FOLDER = 'preproc'     # inside models (or config - sync with save-preproc code below)

In [ ]:
import os
from datetime import datetime
from joblib import dump as save_fld_handler
from joblib import load as load_fld_handler

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 %matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier

In [21]:
# read data
data_file_full_name = os.path.join(
    PROJECT_FOLDER, 
    DATA_FOLDER, 
    DATASET_NAME
    )

ddf = pd.read_csv(data_file_full_name)

In [22]:
# ipynb let's see it - show the read dataset info
ddf.info()

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


# EDA

In [23]:
flds_list = ddf.columns
cat_thres = 9                  # unique values limit for categorical fields (tuning)
num_flds = []
cat_flds = []
bin_flds = []

for i in range(len(flds_list)):
    print(i,':', end=' ')
    fld = flds_list[i]
    view = ddf[fld].value_counts()
    print(view[:cat_thres])
    
    if len(view) > cat_thres:
        print('(etc)')
        num_flds.append(fld)
    elif len(view) == 2:
        print('(binary)')
        bin_flds.append(fld)
    else:
        ddf[fld] = ddf[fld].map(lambda n: fld + '_' + str(n))
        cat_flds.append(fld)
    
    print('-'*50)
    

0 : age
58    68
57    57
54    53
59    46
52    43
51    39
56    39
62    37
60    37
Name: count, dtype: int64
(etc)
--------------------------------------------------
1 : sex
1    713
0    312
Name: count, dtype: int64
(binary)
--------------------------------------------------
2 : cp
0    497
2    284
1    167
3     77
Name: count, dtype: int64
--------------------------------------------------
3 : trestbps
120    128
130    123
140    107
110     64
150     55
138     45
128     39
125     38
160     36
Name: count, dtype: int64
(etc)
--------------------------------------------------
4 : chol
204    21
234    21
197    19
212    18
254    17
269    16
177    14
282    14
240    14
Name: count, dtype: int64
(etc)
--------------------------------------------------
5 : fbs
0    872
1    153
Name: count, dtype: int64
(binary)
--------------------------------------------------
6 : restecg
1    513
0    497
2     15
Name: count, dtype: int64
------------------------------------------

In [24]:
# ipynb let's see it - all categorical fields should become string-types
print('numerical  :', num_flds)
print('categorical:', cat_flds)
print('binary     :', bin_flds)
print()
ddf.info()

numerical  : ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical: ['cp', 'restecg', 'slope', 'ca', 'thal']
binary     : ['sex', 'fbs', 'exang', 'target']

<class 'pandas.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   str    
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   str    
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   str    
 11  ca        1025 non-null   str    
 12  thal      1025 non-null   str    
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(8), str(5)
memory usage: 112.2 KB


# PreProcessing

## Numerical fields

In [25]:
# ipynb let's see it - should be of different scales
ddf[num_flds].std(ddof=0, axis=0)

age          9.067864
trestbps    17.508171
chol        51.567337
thalach     22.994499
oldpeak      1.174480
dtype: float64

In [26]:
scaler = StandardScaler()
X_num = scaler.fit_transform(ddf[num_flds])

scaler_save_file_name = os.path.join(
    PROJECT_FOLDER, 
    MODELS_FOLDER,
    PREPROC_FOLDER, 
    'num_flds_scaler_save.bin'
)

save_fld_handler(scaler, scaler_save_file_name, compress=True)

['..\\models\\preproc\\num_flds_scaler_save.bin']

In [27]:
# ipynb let's see it - should be 5 ones
X_num.std(ddof=0, axis=0)

array([1., 1., 1., 1., 1.])

In [28]:
# ipynb let's see it - should be 5 zeros
X_num.mean(axis=0).round(decimals=13)

array([-0., -0., -0., -0., -0.])

In [29]:
# ipynb let's see it - should be 5 zeros
new_scaler = load_fld_handler(scaler_save_file_name)
(new_scaler.inverse_transform(X_num, copy=True).std(ddof=0, axis=0) - 
 ddf[num_flds].values.std(ddof=0, axis=0)).round(decimals=13)

array([0., 0., 0., 0., 0.])

## Categorical fields

In [30]:
ohe_coder = OneHotEncoder()

X_cat = ohe_coder.fit_transform(ddf[cat_flds])

ohe_save_file_name = os.path.join(
    PROJECT_FOLDER, 
    MODELS_FOLDER,
    PREPROC_FOLDER, 
    'cat_flds_ohe_save.bin'
)

save_fld_handler(ohe_coder, ohe_save_file_name, compress=True)

['..\\models\\preproc\\cat_flds_ohe_save.bin']

In [31]:
# ipynb let's see it - should be True
( X_cat.shape[0] == len(ddf) ) and ( X_cat.shape[0] > len(cat_flds) )

True

In [32]:
# ipynb let's see it - should be zero
new_ohe_coder = load_fld_handler(ohe_save_file_name)
(new_ohe_coder.inverse_transform(X_cat) != ddf[cat_flds].values).sum()

np.int64(0)

## assemble preprocessed X-matrix

In [33]:
X = np.concatenate( [ X_num, X_cat.todense().A, ddf[bin_flds].map(float).values[:,:-1] ], 
                    axis=1).copy()
y = ddf[bin_flds].values[:,-1].copy()

In [34]:
# ipynb let's see it - show the predictors-target shapes and types as well as values of Y
print('X:',X.shape, X.dtype, ' | y:', y.shape, y.dtype, np.unique(y))

X: (1025, 27) float64  | y: (1025,) int64 [0 1]


# train-test split

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# model training

In [ ]:
model_short_name = 'GBC'                 # select model here
model = GradientBoostingClassifier()     # select model here
model.fit(X_train, y_train)

model_save_file_name = os.path.join(
    PROJECT_FOLDER, 
    MODELS_FOLDER, 
    'model_save_' + model_short_name + '-' + str(datetime.now()).split(' ')[0] + '.bin'
)

save_fld_handler(model, model_save_file_name, compress=True)

['..\\models\\model_save_GBC-2026-03-16.bin']

# inferance on new data 

In [38]:
new_model = load_fld_handler(model_save_file_name)
new_model.score(X_train, y_train), new_model.score(X_test, y_test)

(0.9890243902439024, 0.9560975609756097)